# Shared Functions

Reusable utilities for the E-SMAT data engineering pipeline.

This notebook centralizes functions shared by the ingestion, transformation, data-quality, and Delta-load notebooks. It does not execute the pipeline by itself and does not contain credentials, production identifiers, or environment-specific paths.

## Responsibilities

- Microsoft Graph request handling and pagination
- SharePoint field extraction and normalization
- HTML and date cleaning
- Source-record schema validation
- Spark DataFrame creation and column standardization
- Reusable DataFrame quality checks
- Processing metadata

> In Databricks, load this notebook from another notebook with `%run ./01_shared_functions`.


## 1. Imports

Only reusable library imports belong in this notebook. Dependency installation and Python restart commands should remain in the ingestion notebook, because a shared notebook must not restart the active session when loaded with `%run`.


In [ ]:
import html
import re
from datetime import datetime
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence

import requests
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType


## 2. Shared constants

Environment-specific values such as catalog names, SharePoint list names, credentials, and Unity Catalog paths are intentionally excluded.


In [ ]:
GRAPH_BASE_URL = "https://graph.microsoft.com/v1.0"
DEFAULT_REQUEST_TIMEOUT_SECONDS = 60
DEFAULT_PROCESSING_TIMEZONE = "America/Sao_Paulo"

MONTHS_MAP = {
    # Portuguese
    "janeiro": "01",
    "fevereiro": "02",
    "março": "03",
    "abril": "04",
    "maio": "05",
    "junho": "06",
    "julho": "07",
    "agosto": "08",
    "setembro": "09",
    "outubro": "10",
    "novembro": "11",
    "dezembro": "12",
    # Spanish
    "enero": "01",
    "febrero": "02",
    "marzo": "03",
    "abril": "04",
    "mayo": "05",
    "junio": "06",
    "julio": "07",
    "agosto": "08",
    "septiembre": "09",
    "octubre": "10",
    "noviembre": "11",
    "diciembre": "12",
    # English
    "january": "01",
    "february": "02",
    "march": "03",
    "april": "04",
    "may": "05",
    "june": "06",
    "july": "07",
    "august": "08",
    "september": "09",
    "october": "10",
    "november": "11",
    "december": "12",
}


## 3. Microsoft Graph helpers

These functions receive authentication headers as parameters. The shared notebook does not create or store access tokens.


In [ ]:
def request_graph_json(
    url: str,
    request_headers: Mapping[str, str],
    timeout_seconds: int = DEFAULT_REQUEST_TIMEOUT_SECONDS,
) -> Dict[str, Any]:
    """Send a GET request to Microsoft Graph and return decoded JSON."""
    response = requests.get(
        url,
        headers=dict(request_headers),
        timeout=timeout_seconds,
    )

    try:
        response.raise_for_status()
    except requests.HTTPError as exc:
        raise RuntimeError(
            "Microsoft Graph request failed. "
            f"Status: {response.status_code}. "
            f"Endpoint: {url}. "
            f"Response: {response.text[:1000]}"
        ) from exc

    try:
        return response.json()
    except requests.JSONDecodeError as exc:
        raise ValueError(
            f"Microsoft Graph returned a non-JSON response for: {url}"
        ) from exc


def get_paginated_graph_values(
    initial_url: str,
    request_headers: Mapping[str, str],
    timeout_seconds: int = DEFAULT_REQUEST_TIMEOUT_SECONDS,
) -> List[Dict[str, Any]]:
    """Collect all values from a paginated Microsoft Graph endpoint."""
    current_url = initial_url
    values: List[Dict[str, Any]] = []

    while current_url:
        page = request_graph_json(
            url=current_url,
            request_headers=request_headers,
            timeout_seconds=timeout_seconds,
        )
        values.extend(page.get("value", []))
        current_url = page.get("@odata.nextLink")

    return values


## 4. SharePoint value cleaning

The functions below convert common Microsoft Graph and SharePoint representations into stable text values before Spark DataFrame creation.


In [ ]:
def clean_html(value: Any) -> Optional[str]:
    """Remove HTML tags, decode entities, and normalize whitespace."""
    if value is None:
        return None

    text = html.unescape(str(value))
    text = re.sub(r"<[^>]+>", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def format_date_to_ddmmyyyy(value: Any) -> Optional[str]:
    """Convert supported textual date representations to DD/MM/YYYY."""
    if value is None or not str(value).strip():
        return None

    value_text = str(value).strip()
    normalized_text = value_text.lower()

    for month_name, month_number in MONTHS_MAP.items():
        if month_name in normalized_text:
            numeric_parts = re.findall(r"\d+", value_text)
            if len(numeric_parts) >= 2:
                day = numeric_parts[0].zfill(2)
                year = numeric_parts[-1]
                if len(year) == 2:
                    year = f"20{year}"
                return f"{day}/{month_number}/{year}"

    cleaned_value = value_text.replace("Z", "").replace("T", " ").split(" ")[0]

    for date_format in ("%Y-%m-%d", "%d-%m-%Y", "%d/%m/%Y", "%Y/%m/%d"):
        try:
            return datetime.strptime(cleaned_value, date_format).strftime("%d/%m/%Y")
        except ValueError:
            continue

    return value_text


def extract_sharepoint_value(value: Any) -> Optional[str]:
    """Extract readable text from common Graph and SharePoint objects."""
    if value is None:
        return None

    if isinstance(value, dict):
        for key in (
            "displayName",
            "Title",
            "name",
            "email",
            "lookupValue",
            "text",
        ):
            extracted_value = value.get(key)
            if extracted_value not in (None, ""):
                return str(extracted_value)
        return str(value)

    if isinstance(value, list):
        extracted_values = [extract_sharepoint_value(item) for item in value]
        return ", ".join(item for item in extracted_values if item)

    return str(value)


def get_field_flexible(
    fields: Mapping[str, Any],
    column_name: str,
    field_aliases: Optional[Mapping[str, str]] = None,
) -> Any:
    """Retrieve a field through direct, alias, case-insensitive, or normalized matching."""
    if not fields or not column_name:
        return None

    aliases = field_aliases or {}

    if column_name.lower() == "author":
        for candidate in (
            "Author",
            "AuthorLookupId",
            "Editor",
            "CreatedBy",
            "AuthorId",
        ):
            for key, value in fields.items():
                if key.lower() == candidate.lower():
                    return value

    for target_key in (column_name, aliases.get(column_name)):
        if not target_key:
            continue
        if target_key in fields:
            return fields[target_key]
        for key, value in fields.items():
            if key.lower() == target_key.lower():
                return value

    normalized_target = re.sub(r"[^a-z0-9]", "", column_name.lower())
    for key, value in fields.items():
        normalized_key = re.sub(r"[^a-z0-9]", "", key.lower())
        if normalized_key == normalized_target:
            return value

    return None


def format_field_value(column_name: str, value: Any) -> Optional[str]:
    """Standardize a SharePoint field before DataFrame creation."""
    if value is None:
        return None

    if isinstance(value, bool):
        return "Sim" if value else "Não"

    normalized_value = str(value).strip().lower()
    if normalized_value in {"true", "false"}:
        return "Sim" if normalized_value == "true" else "Não"

    normalized_column = column_name.lower()
    if "date" in normalized_column or "data" in normalized_column:
        return format_date_to_ddmmyyyy(value)

    processed_value = extract_sharepoint_value(value)
    text_patterns = (
        "description",
        "descricao",
        "observation",
        "observacao",
        "question",
        "questao",
    )

    if any(pattern in normalized_column for pattern in text_patterns):
        return clean_html(processed_value)

    return processed_value


## 5. Record extraction and schema validation

The source-record schema is validated before the Spark DataFrame is created. This prevents missing, extra, or duplicate columns from advancing silently through the pipeline.


In [ ]:
def transform_sharepoint_items(
    sharepoint_items: Sequence[Mapping[str, Any]],
    source_columns: Sequence[str],
    field_aliases: Optional[Mapping[str, str]] = None,
) -> List[Dict[str, Optional[str]]]:
    """Extract and format configured fields from SharePoint list items."""
    transformed_records = []

    for item in sharepoint_items:
        fields = item.get("fields") or {}
        transformed_record = {}

        for column_name in source_columns:
            raw_value = get_field_flexible(
                fields=fields,
                column_name=column_name,
                field_aliases=field_aliases,
            )
            transformed_record[column_name] = format_field_value(
                column_name=column_name,
                value=raw_value,
            )

        transformed_records.append(transformed_record)

    return transformed_records


def validate_records_schema(
    records: List[Dict[str, Any]],
    source_columns: Sequence[str],
    rename_mapping: Mapping[str, str],
    dataset_name: str,
) -> Dict[str, Any]:
    """Validate configured and extracted schemas before DataFrame creation."""
    if not isinstance(records, list):
        raise ValueError(f"Dataset '{dataset_name}': records must be a list.")
    if not source_columns:
        raise ValueError(f"Dataset '{dataset_name}': no columns were configured.")
    if len(source_columns) != len(set(source_columns)):
        raise ValueError(f"Dataset '{dataset_name}': duplicate source columns.")

    configured_columns = set(source_columns)
    unknown_mapping_columns = sorted(set(rename_mapping) - configured_columns)
    if unknown_mapping_columns:
        raise ValueError(
            f"Dataset '{dataset_name}': mapping contains unconfigured fields: "
            f"{unknown_mapping_columns}"
        )

    target_columns = list(rename_mapping.values())
    if len(target_columns) != len(set(target_columns)):
        raise ValueError(
            f"Dataset '{dataset_name}': duplicate standardized target columns."
        )

    errors = []
    for record_index, record in enumerate(records):
        if not isinstance(record, dict):
            errors.append(f"record {record_index} is not a dictionary")
            continue

        missing_columns = sorted(configured_columns - set(record))
        extra_columns = sorted(set(record) - configured_columns)

        if missing_columns:
            errors.append(f"record {record_index} missing {missing_columns}")
        if extra_columns:
            errors.append(f"record {record_index} has extra {extra_columns}")

    if errors:
        raise ValueError(
            f"Schema validation failed for '{dataset_name}': "
            + "; ".join(errors[:10])
        )

    null_counts = {
        column_name: sum(record.get(column_name) is None for record in records)
        for column_name in source_columns
    }
    fully_null_columns = [
        column_name
        for column_name, null_count in null_counts.items()
        if records and null_count == len(records)
    ]

    if fully_null_columns:
        print(
            f"Warning: '{dataset_name}' contains fully null columns: "
            f"{fully_null_columns}"
        )

    return {
        "dataset": dataset_name,
        "valid": True,
        "record_count": len(records),
        "source_column_count": len(source_columns),
        "fully_null_columns": fully_null_columns,
        "null_counts": null_counts,
    }


## 6. Spark DataFrame helpers

In [ ]:
def create_string_dataframe(
    spark_session,
    records: List[Dict[str, Any]],
    source_columns: Sequence[str],
    dataset_name: str,
) -> DataFrame:
    """Create a nullable string DataFrame with deterministic column order."""
    if not records:
        raise ValueError(f"Dataset '{dataset_name}' contains no records.")

    source_columns = list(source_columns)
    schema = StructType(
        [StructField(column_name, StringType(), True) for column_name in source_columns]
    )
    ordered_records = [
        tuple(record.get(column_name) for column_name in source_columns)
        for record in records
    ]

    dataframe = spark_session.createDataFrame(ordered_records, schema=schema)

    if dataframe.columns != source_columns:
        raise ValueError(
            f"Dataset '{dataset_name}': generated source schema differs from "
            "the configured schema."
        )

    return dataframe


def apply_column_mapping(
    dataframe: DataFrame,
    rename_mapping: Mapping[str, str],
) -> DataFrame:
    """Rename source fields to standardized analytical names."""
    result = dataframe

    for source_column, target_column in rename_mapping.items():
        if source_column in result.columns and source_column != target_column:
            result = result.withColumnRenamed(source_column, target_column)

    return result


def validate_renamed_dataframe(
    dataframe: DataFrame,
    source_columns: Sequence[str],
    rename_mapping: Mapping[str, str],
    dataset_name: str,
) -> DataFrame:
    """Validate the standardized DataFrame schema after renaming."""
    expected_columns = [
        rename_mapping.get(source_column, source_column)
        for source_column in source_columns
    ]

    if dataframe.columns != expected_columns:
        raise ValueError(
            f"Dataset '{dataset_name}': standardized schema mismatch. "
            f"Expected {expected_columns}; generated {dataframe.columns}."
        )

    if len(dataframe.columns) != len(set(dataframe.columns)):
        raise ValueError(
            f"Dataset '{dataset_name}': duplicate standardized columns."
        )

    return dataframe


def require_columns(
    dataframe: DataFrame,
    required_columns: Iterable[str],
    dataset_name: str,
) -> None:
    """Fail when a DataFrame does not contain required columns."""
    missing_columns = sorted(set(required_columns) - set(dataframe.columns))
    if missing_columns:
        raise ValueError(
            f"Dataset '{dataset_name}' is missing required columns: "
            f"{missing_columns}"
        )


def add_processing_timestamp(
    dataframe: DataFrame,
    timezone: str = DEFAULT_PROCESSING_TIMEZONE,
) -> DataFrame:
    """Add the pipeline processing timestamp in the configured timezone."""
    return dataframe.withColumn(
        "processing_timestamp",
        F.from_utc_timestamp(F.current_timestamp(), timezone),
    )


## 7. Shared notebook status

This final cell provides a simple confirmation when the notebook is loaded by another Databricks notebook.


In [ ]:
print("Shared E-SMAT pipeline functions loaded successfully.")
